# Day 19 - Financial Trend & Momentum Analysis

## Objectives

- Analyze company financial performance across multiple years
- Calculate year-over-year changes in financial ratios
- Measure financial momentum
- Identify improving and declining companies
- Generate trend-based analytical reports

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

REPORT_DIR = "../reports"

print("Day 19 - Financial Trend Analysis")

Day 19 - Financial Trend Analysis


In [2]:
ratio_path = "../reports/financial_ratios_day12.csv"

ratios = pd.read_csv(ratio_path)

print("Shape:", ratios.shape)
print("Columns:")
print(ratios.columns.tolist())

ratios.head()

Shape: (1184, 7)
Columns:
['company_id', 'year', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'dividend_payout_ratio']


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [3]:
duplicate_count = ratios.duplicated(
    subset=["company_id", "year"]
).sum()

print("Duplicate Company-Year Records:", duplicate_count)

Duplicate Company-Year Records: 119


In [4]:
ratios = ratios.drop_duplicates(
    subset=["company_id", "year"]
).copy()

print("Shape After Removing Duplicates:", ratios.shape)

Shape After Removing Duplicates: (1065, 7)


In [5]:
ratios["financial_year"] = (
    ratios["year"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

ratios["financial_year"] = pd.to_numeric(
    ratios["financial_year"],
    errors="coerce"
)

ratios[
    ["company_id", "year", "financial_year"]
].head(10)

,company_id,year,financial_year
0,ABB,Dec 2012,2012
1,ABB,Mar 2014,2014
3,ABB,Mar 2015,2015
5,ABB,Mar 2016,2016
7,ABB,Mar 2017,2017
9,ABB,Mar 2018,2018
11,ABB,Mar 2019,2019
13,ABB,Mar 2020,2020
15,ABB,Mar 2021,2021
17,ABB,Mar 2022,2022


In [6]:
ratios = ratios.sort_values(
    by=["company_id", "financial_year"]
).reset_index(drop=True)

ratios[
    ["company_id", "year", "financial_year"]
].head(15)

,company_id,year,financial_year
0,ABB,Dec 2012,2012
1,ABB,Mar 2014,2014
2,ABB,Mar 2015,2015
3,ABB,Mar 2016,2016
4,ABB,Mar 2017,2017
5,ABB,Mar 2018,2018
6,ABB,Mar 2019,2019
7,ABB,Mar 2020,2020
8,ABB,Mar 2021,2021
9,ABB,Mar 2022,2022


In [7]:
possible_metrics = [
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "dividend_payout_ratio"
]

available_metrics = [
    col
    for col in possible_metrics
    if col in ratios.columns
]

print("Available Trend Metrics:")
print(available_metrics)

Available Trend Metrics:
['return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'dividend_payout_ratio']


In [8]:
for col in available_metrics:
    ratios[col] = pd.to_numeric(
        ratios[col],
        errors="coerce"
    )

print(ratios[available_metrics].dtypes)

return_on_equity_pct     float64
debt_to_equity           float64
interest_coverage        float64
asset_turnover           float64
dividend_payout_ratio    float64
dtype: object


In [9]:
for metric in available_metrics:
    ratios[f"{metric}_yoy_change"] = (
        ratios
        .groupby("company_id")[metric]
        .diff()
    )

yoy_columns = [
    f"{metric}_yoy_change"
    for metric in available_metrics
]

ratios[
    ["company_id", "financial_year"]
    + available_metrics
    + yoy_columns
].head(15)

,company_id,financial_year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,return_on_equity_pct_yoy_change,debt_to_equity_yoy_change,interest_coverage_yoy_change,asset_turnover_yoy_change,dividend_payout_ratio_yoy_change
0,ABB,2012,22.411128,0.000000,NaN,1.822492,17.241379,NaN,NaN,NaN,NaN,NaN
1,ABB,2014,25.126904,0.000000,NaN,1.998244,12.626263,2.715775,0.000000,NaN,0.175752,-4.615117
2,ABB,2015,24.439701,0.000000,NaN,1.665939,12.663755,-0.687202,0.000000,NaN,-0.332305,0.037493
3,ABB,2016,21.338912,0.000000,121.666667,1.617574,11.372549,-3.100789,0.000000,NaN,-0.048365,-1.291206
4,ABB,2017,19.971161,0.000000,199.000000,1.405131,11.191336,-1.367751,0.000000,77.333333,-0.212444,-0.181213
5,ABB,2018,23.685765,0.000000,131.250000,1.365066,7.231920,3.714604,0.000000,-67.750000,-0.040064,-3.959416
6,ABB,2019,22.410359,0.000000,302.500000,1.250935,6.888889,-1.275406,0.000000,171.250000,-0.114131,-0.343031
7,ABB,2020,24.393254,0.071987,84.111111,1.153933,15.177066,1.982895,0.071987,-218.388889,-0.097002,8.288177
8,ABB,2021,26.556495,0.058801,51.222222,1.122396,12.301013,2.163241,-0.013186,-32.888889,-0.031537,-2.876053
9,ABB,2022,28.333333,0.053901,56.947368,1.163116,9.136421,1.776838,-0.004900,5.725146,0.040720,-3.164592


In [10]:
if "return_on_equity_pct_yoy_change" in ratios.columns:
    ratios["roe_momentum"] = np.select(
        [
            ratios["return_on_equity_pct_yoy_change"] > 2,
            ratios["return_on_equity_pct_yoy_change"] < -2
        ],
        [
            1,
            -1
        ],
        default=0
    )

ratios[
    [
        "company_id",
        "financial_year",
        "return_on_equity_pct",
        "return_on_equity_pct_yoy_change",
        "roe_momentum"
    ]
].head(15)

,company_id,financial_year,return_on_equity_pct,return_on_equity_pct_yoy_change,roe_momentum
0,ABB,2012,22.411128,NaN,0
1,ABB,2014,25.126904,2.715775,1
2,ABB,2015,24.439701,-0.687202,0
3,ABB,2016,21.338912,-3.100789,-1
4,ABB,2017,19.971161,-1.367751,0
5,ABB,2018,23.685765,3.714604,1
6,ABB,2019,22.410359,-1.275406,0
7,ABB,2020,24.393254,1.982895,0
8,ABB,2021,26.556495,2.163241,1
9,ABB,2022,28.333333,1.776838,0


In [11]:
if "debt_to_equity_yoy_change" in ratios.columns:
    ratios["debt_momentum"] = np.select(
        [
            ratios["debt_to_equity_yoy_change"] < -0.1,
            ratios["debt_to_equity_yoy_change"] > 0.1
        ],
        [
            1,
            -1
        ],
        default=0
    )

ratios[
    [
        "company_id",
        "financial_year",
        "debt_to_equity",
        "debt_to_equity_yoy_change",
        "debt_momentum"
    ]
].head(15)

,company_id,financial_year,debt_to_equity,debt_to_equity_yoy_change,debt_momentum
0,ABB,2012,0.000000,NaN,0
1,ABB,2014,0.000000,0.000000,0
2,ABB,2015,0.000000,0.000000,0
3,ABB,2016,0.000000,0.000000,0
4,ABB,2017,0.000000,0.000000,0
5,ABB,2018,0.000000,0.000000,0
6,ABB,2019,0.000000,0.000000,0
7,ABB,2020,0.071987,0.071987,0
8,ABB,2021,0.058801,-0.013186,0
9,ABB,2022,0.053901,-0.004900,0


In [12]:
if "interest_coverage_yoy_change" in ratios.columns:
    ratios["interest_momentum"] = np.select(
        [
            ratios["interest_coverage_yoy_change"] > 1,
            ratios["interest_coverage_yoy_change"] < -1
        ],
        [
            1,
            -1
        ],
        default=0
    )

In [13]:
if "asset_turnover_yoy_change" in ratios.columns:
    ratios["asset_turnover_momentum"] = np.select(
        [
            ratios["asset_turnover_yoy_change"] > 0.1,
            ratios["asset_turnover_yoy_change"] < -0.1
        ],
        [
            1,
            -1
        ],
        default=0
    )

In [14]:
momentum_columns = [
    col
    for col in [
        "roe_momentum",
        "debt_momentum",
        "interest_momentum",
        "asset_turnover_momentum"
    ]
    if col in ratios.columns
]

print("Momentum Columns:")
print(momentum_columns)

Momentum Columns:
['roe_momentum', 'debt_momentum', 'interest_momentum', 'asset_turnover_momentum']


In [15]:
ratios["financial_momentum_score"] = (
    ratios[momentum_columns]
    .sum(axis=1)
)

ratios[
    [
        "company_id",
        "financial_year"
    ]
    + momentum_columns
    + ["financial_momentum_score"]
].head(15)

,company_id,financial_year,roe_momentum,debt_momentum,interest_momentum,asset_turnover_momentum,financial_momentum_score
0,ABB,2012,0,0,0,0,0
1,ABB,2014,1,0,0,1,2
2,ABB,2015,0,0,0,-1,-1
3,ABB,2016,-1,0,0,0,-1
4,ABB,2017,0,0,1,-1,0
5,ABB,2018,1,0,-1,0,0
6,ABB,2019,0,0,1,-1,0
7,ABB,2020,0,0,-1,0,-1
8,ABB,2021,1,0,-1,0,0
9,ABB,2022,0,0,1,0,1


In [16]:
ratios["trend_classification"] = pd.cut(
    ratios["financial_momentum_score"],
    bins=[
        -np.inf,
        -2,
        0,
        2,
        np.inf
    ],
    labels=[
        "Strong Decline",
        "Declining",
        "Improving",
        "Strong Improvement"
    ],
    include_lowest=True
)

ratios[
    [
        "company_id",
        "financial_year",
        "financial_momentum_score",
        "trend_classification"
    ]
].head(20)

,company_id,financial_year,financial_momentum_score,trend_classification
0,ABB,2012,0,Declining
1,ABB,2014,2,Improving
2,ABB,2015,-1,Declining
3,ABB,2016,-1,Declining
4,ABB,2017,0,Declining
5,ABB,2018,0,Declining
6,ABB,2019,0,Declining
7,ABB,2020,-1,Declining
8,ABB,2021,0,Declining
9,ABB,2022,1,Improving


In [17]:
latest_company_trends = (
    ratios
    .sort_values(
        ["company_id", "financial_year"]
    )
    .groupby("company_id")
    .tail(1)
    .copy()
)

latest_company_trends[
    [
        "company_id",
        "financial_year",
        "financial_momentum_score",
        "trend_classification"
    ]
].head(20)

,company_id,financial_year,financial_momentum_score,trend_classification
11,ABB,2024,2,Improving
22,ADANIENSOL,2024,0,Declining
34,ADANIENT,2024,-1,Declining
42,ADANIGREEN,2024,1,Improving
54,ADANIPORTS,2024,3,Strong Improvement
66,ADANIPOWER,2024,3,Strong Improvement
77,AMBUJACEM,2024,-3,Strong Decline
89,APOLLOHOSP,2024,0,Declining
101,ASIANPAINT,2024,-1,Declining
113,AXISBANK,2024,2,Improving


In [18]:
top_improving_companies = (
    latest_company_trends[
        [
            "company_id",
            "financial_year",
            "financial_momentum_score",
            "trend_classification"
        ]
    ]
    .sort_values(
        "financial_momentum_score",
        ascending=False
    )
    .head(20)
)

top_improving_companies

,company_id,financial_year,financial_momentum_score,trend_classification
1028,TRENT,2024,4,Strong Improvement
944,TATAMOTORS,2024,4,Strong Improvement
66,ADANIPOWER,2024,3,Strong Improvement
54,ADANIPORTS,2024,3,Strong Improvement
825,PNB,2024,3,Strong Improvement
669,LODHA,2024,3,Strong Improvement
183,BEL,2024,3,Strong Improvement
255,CANBK,2024,2,Improving
243,BRITANNIA,2024,2,Improving
231,BPCL,2024,2,Improving


In [19]:
declining_companies = (
    latest_company_trends[
        latest_company_trends[
            "financial_momentum_score"
        ] < 0
    ][
        [
            "company_id",
            "financial_year",
            "financial_momentum_score",
            "trend_classification"
        ]
    ]
    .sort_values(
        "financial_momentum_score"
    )
)

declining_companies.head(20)

,company_id,financial_year,financial_momentum_score,trend_classification
77,AMBUJACEM,2024,-3,Strong Decline
387,GODREJCP,2024,-3,Strong Decline
968,TATASTEEL,2024,-3,Strong Decline
693,LTIM,2024,-3,Strong Decline
561,INFY,2024,-2,Strong Decline
583,IRCTC,2024,-2,Strong Decline
315,DIVISLAB,2024,-2,Strong Decline
291,COALINDIA,2024,-2,Strong Decline
630,JSWENERGY,2024,-2,Strong Decline
660,LICI,2024,-2,Strong Decline


In [20]:
trend_distribution = (
    latest_company_trends[
        "trend_classification"
    ]
    .value_counts(dropna=False)
    .rename_axis("trend_classification")
    .reset_index(name="company_count")
)

trend_distribution

,trend_classification,company_count
0,Improving,39
1,Declining,33
2,Strong Decline,13
3,Strong Improvement,7


In [21]:
os.makedirs(REPORT_DIR, exist_ok=True)

ratios.to_csv(
    "../reports/day19_financial_trends.csv",
    index=False
)

latest_company_trends.to_csv(
    "../reports/day19_latest_company_trends.csv",
    index=False
)

top_improving_companies.to_csv(
    "../reports/day19_top_improving_companies.csv",
    index=False
)

declining_companies.to_csv(
    "../reports/day19_declining_companies.csv",
    index=False
)

trend_distribution.to_csv(
    "../reports/day19_trend_distribution.csv",
    index=False
)

print("Day 19 reports saved successfully.")

Day 19 reports saved successfully.


In [22]:
print("=" * 60)
print("DAY 19 - FINAL VALIDATION")
print("=" * 60)

print("Trend Dataset Shape:", ratios.shape)
print("Companies Analyzed:", ratios["company_id"].nunique())
print("Momentum Metrics:", momentum_columns)

print("\nTREND DISTRIBUTION")
print(trend_distribution)

print("\nTOP IMPROVING COMPANIES")
print(top_improving_companies.head(10))

print("\nDAY 19 COMPLETED SUCCESSFULLY")

DAY 19 - FINAL VALIDATION
Trend Dataset Shape: (1065, 19)
Companies Analyzed: 92
Momentum Metrics: ['roe_momentum', 'debt_momentum', 'interest_momentum', 'asset_turnover_momentum']

TREND DISTRIBUTION
  trend_classification  company_count
0            Improving             39
1            Declining             33
2       Strong Decline             13
3   Strong Improvement              7

TOP IMPROVING COMPANIES
      company_id  financial_year  financial_momentum_score trend_classification
1028       TRENT            2024                         4   Strong Improvement
944   TATAMOTORS            2024                         4   Strong Improvement
66    ADANIPOWER            2024                         3   Strong Improvement
54    ADANIPORTS            2024                         3   Strong Improvement
825          PNB            2024                         3   Strong Improvement
669        LODHA            2024                         3   Strong Improvement
183          BEL        